# Aritmética modular

**Matemática Discreta (2026/27)** · Edgar Martínez-Moro · Departamento de Matemática Aplicada, Escuela de Ingeniería Informática, Universidad de Valladolid

Este cuaderno acompaña a la sesión 19 del tema de **teoría de grafos**. Contiene:

1. La división entera (y una diferencia con el operador `%` de Python).
2. El algoritmo de Euclides y la identidad de Bézout.
3. Ecuaciones diofánticas.
4. Congruencias: reducir, ecuaciones de congruencia lineales e inversos.
5. Potencias módulo $m$ por cuadrados sucesivos y el pequeño teorema de Fermat.
6. Sistemas de congruencias: el teorema chino de los restos.

## Repaso

- **División entera:** $a=bq+r$ con $0\le r<|b|$.
- **Algoritmo de Euclides:** $\operatorname{mcd}(a,b)$ es el último resto no nulo de las divisiones sucesivas; además $\operatorname{mcd}(a,b)=ax+by$ para ciertos enteros $x,y$ (Bézout).
- **Ecuación diofántica** $ax+by=c$: tiene solución si y sólo si $\operatorname{mcd}(a,b)\mid c$.
- **Congruencia:** $a\equiv b\bmod m$ si $m\mid a-b$. Es una relación de equivalencia con $m$ clases (sesión 12).

In [1]:
from math import floor, ceil

## 1. La división entera

In [2]:
def division(a, b):
    """Cociente y resto con a = b q + r, 0 <= r < |b| (algoritmo 5.3)."""
    if b == 0:
        raise ZeroDivisionError("b no puede ser 0")
    q = floor(a / b) if b > 0 else ceil(a / b)
    return q, a - b * q


def divide(b, a):
    """b | a"""
    return a % b == 0

In [3]:
for a, b in ((9, 4), (-9, 4), (9, -4), (-9, -4)):
    q, r = division(a, b)
    print(f"{a:>3} = {b:>2}·({q:>2}) + {r}     divmod de Python: {divmod(a, b)}")

  9 =  4·( 2) + 1     divmod de Python: (2, 1)
 -9 =  4·(-3) + 3     divmod de Python: (-3, 3)
  9 = -4·(-2) + 1     divmod de Python: (-3, -3)
 -9 = -4·( 3) + 3     divmod de Python: (2, -1)


Con $b<0$, `divmod` y `%` de Python dan un resto con el signo de $b$ (por ejemplo, `-9 % -4` es `-1`), que no cumple $0\le r<|b|$. Con $m>0$, en cambio, `a % m` es siempre el representante canónico, también para $a$ negativo.

In [4]:
print(-9 % 4, 9 % -4, -9 % -4)

3 -3 -1


## 2. Euclides y Bézout

In [5]:
def euclides(a, b, traza=False):
    """mcd(a, b) dividiendo a entre b y, luego, cada divisor entre el resto."""
    a, b = abs(a), abs(b)
    while b != 0:
        q, r = division(a, b)
        if traza:
            print(f"  {a} = {b}·{q} + {r}")
        a, b = b, r
    return a


def euclides_extendido(a, b, traza=False):
    """
    Devuelve (d, x, y) con d = mcd(a, b) = a x + b y.
    Cada resto se escribe como combinación de a y b: si
        r_{i-2} = a x_{i-2} + b y_{i-2},  r_{i-1} = a x_{i-1} + b y_{i-1}
    y r_i = r_{i-2} - q r_{i-1}, entonces x_i = x_{i-2} - q x_{i-1} (igual con y).
    """
    r0, r1 = a, b
    x0, x1 = 1, 0
    y0, y1 = 0, 1
    if traza:
        print(f"  {'q':>4}  {'resto':>6}  {'x':>5}  {'y':>5}")
        print(f"  {'':>4}  {r0:>6}  {x0:>5}  {y0:>5}")
        print(f"  {'':>4}  {r1:>6}  {x1:>5}  {y1:>5}")
    while r1 != 0:
        q, r = division(r0, r1)
        r0, r1 = r1, r
        x0, x1 = x1, x0 - q * x1
        y0, y1 = y1, y0 - q * y1
        if traza:
            print(f"  {q:>4}  {r1:>6}  {x1:>5}  {y1:>5}")
    if r0 < 0:
        r0, x0, y0 = -r0, -x0, -y0
    return r0, x0, y0

In [6]:
print("mcd =", euclides(124, 48, traza=True))
print()
d, x, y = euclides_extendido(124, 48, traza=True)
print(f"mcd(124, 48) = {d} = 124·({x}) + 48·({y})")

  124 = 48·2 + 28
  48 = 28·1 + 20
  28 = 20·1 + 8
  20 = 8·2 + 4
  8 = 4·2 + 0
mcd = 4

     q   resto      x      y
           124      1      0
            48      0      1
     2      28      1     -2
     1      20     -1      3
     1       8      2     -5
     2       4     -5     13
     2       0     12    -31
mcd(124, 48) = 4 = 124·(-5) + 48·(13)


Cada fila de la tabla expresa un resto como $124x+48y$: es el ejemplo 5.9. Comprobamos con la función `gcd` de Python y medimos cuántas divisiones hace Euclides con números grandes: los peores casos son los números de Fibonacci consecutivos (teorema de Lamé).

In [7]:
from math import gcd
import random
random.seed(1)
for _ in range(1000):
    a, b = random.randint(-10**6, 10**6), random.randint(1, 10**6)
    d, x, y = euclides_extendido(a, b)
    assert d == gcd(a, b) and a * x + b * y == d
print("comprobado")

def pasos(a, b):
    n = 0
    while b:
        a, b, n = b, a % b, n + 1
    return n

F = [1, 1]
while len(F) < 40:
    F.append(F[-1] + F[-2])
print("Fibonacci:", F[38], F[37], "->", pasos(F[38], F[37]), "divisiones")
print("al azar:  ", pasos(random.randint(10**7, 10**8), random.randint(10**7, 10**8)), "divisiones")

comprobado
Fibonacci: 63245986 39088169 -> 37 divisiones
al azar:   17 divisiones


## 3. Ecuaciones diofánticas

$ax+by=c$ tiene solución si y sólo si $d=\operatorname{mcd}(a,b)\mid c$; entonces $x=x_0+\frac bd k$, $y=y_0-\frac ad k$.

In [8]:
def diofantica(a, b, c):
    """
    Resuelve a x + b y = c en Z. Devuelve None si no hay solución, o
    (x0, y0, px, py): las soluciones son x = x0 + px k, y = y0 + py k.
    """
    d, p, q = euclides_extendido(a, b)
    if c % d != 0:
        return None
    m = c // d
    return p * m, q * m, b // d, -(a // d)


def fmt_diofantica(sol):
    if sol is None:
        return "sin solución"
    x0, y0, px, py = sol
    s = lambda v: f"+ {v}" if v >= 0 else f"- {-v}"
    return f"x = {x0} {s(px)}k,  y = {y0} {s(py)}k,  k entero"

In [9]:
for a, b, c in ((6, 10, 2), (15, 36, 9), (61, 41, 5), (6, 10, 3)):
    print(f"{a}x + {b}y = {c}:  ", fmt_diofantica(diofantica(a, b, c)))

6x + 10y = 2:   x = 2 + 5k,  y = -1 - 3k,  k entero
15x + 36y = 9:   x = 15 + 12k,  y = -6 - 5k,  k entero
61x + 41y = 5:   x = -10 + 41k,  y = 15 - 61k,  k entero
6x + 10y = 3:   sin solución


Comprobamos unas cuantas soluciones del ejemplo 3, y buscamos la solución con $x$ e $y$ positivos más pequeños de $61x+41y=1000$:

In [10]:
x0, y0, px, py = diofantica(61, 41, 5)
for k in range(-2, 3):
    x, y = x0 + px * k, y0 + py * k
    print(k, (x, y), 61 * x + 41 * y)
x0, y0, px, py = diofantica(61, 41, 1000)
print([(x0 + px * k, y0 + py * k) for k in range(-100, 100) if x0 + px * k > 0 and y0 + py * k > 0])

-2 (-92, 137) 5
-1 (-51, 76) 5
0 (-10, 15) 5
1 (31, -46) 5
2 (72, -107) 5
[(9, 11)]


## 4. Congruencias

### Reducir y clases

In [11]:
def reducir(a, m):
    """Representante canónico de [a]_m, en {0, 1, ..., m-1}."""
    return a % m          # en Python, con m > 0, el resto ya está entre 0 y m-1

In [12]:
print([reducir(a, 7) for a in (-15, -1, 0, 6, 7, 100)])
m = 5
print({r: [a for a in range(-12, 13) if reducir(a, m) == r] for r in range(m)})

[6, 6, 0, 6, 0, 2]
{0: [-10, -5, 0, 5, 10], 1: [-9, -4, 1, 6, 11], 2: [-8, -3, 2, 7, 12], 3: [-12, -7, -2, 3, 8], 4: [-11, -6, -1, 4, 9]}


La proposición 5.19: si $m=kn$, la clase $[a]_n$ es la unión de $k$ clases módulo $m$. Por ejemplo, $[15]_{41}=[15]_{123}\cup[56]_{123}\cup[97]_{123}$:

In [13]:
print(sorted({reducir(x, 123) for x in range(-1000, 1000) if reducir(x, 41) == 15}))

[15, 56, 97]


### Ecuaciones de congruencia lineales (teorema 5.21)

In [14]:
def congruencia_lineal(a, b, m):
    """
    Resuelve a x ≡ b (mod m) (teorema 5.21). Devuelve None si no hay
    solución, o (x0, m') con la solución [x0]_{m'}, m' = m / mcd(a, m).
    """
    d, p, _ = euclides_extendido(a, m)
    if b % d != 0:
        return None
    m2 = m // d
    return reducir(p * (b // d), m2), m2


def inverso(a, m):
    """Inverso de a módulo m (existe si mcd(a, m) = 1)."""
    sol = congruencia_lineal(a, 1, m)
    if sol is None:
        raise ValueError(f"{a} no tiene inverso módulo {m}")
    return sol[0]

In [15]:
for a, b, m in ((183, 39, 123), (2, 12, 23), (6, 210, 20), (6, 7, 20)):
    print(f"{a}x ≡ {b} (mod {m}):", congruencia_lineal(a, b, m))
print("inversos módulo 7:", {a: inverso(a, 7) for a in range(1, 7)})

183x ≡ 39 (mod 123): (15, 41)
2x ≡ 12 (mod 23): (6, 23)
6x ≡ 210 (mod 20): (5, 10)
6x ≡ 7 (mod 20): None
inversos módulo 7: {1: 1, 2: 4, 3: 5, 4: 2, 5: 3, 6: 6}


Comprobación por fuerza bruta de $183x\equiv39 \bmod 123$:

In [16]:
print([x for x in range(123) if (183 * x - 39) % 123 == 0])

[15, 56, 97]


## 5. Potencias módulo $m$

Por cuadrados sucesivos: se escribe el exponente en binario y se multiplican las potencias $a^{2^i}$ que hacen falta, reduciendo siempre. Son unas $2\log_2 k$ multiplicaciones.

In [17]:
def potencia_mod(a, k, m, traza=False):
    """
    a^k mod m por cuadrados sucesivos: se escribe k en binario y se
    multiplican las potencias a^(2^i) que hacen falta. Unas 2 log2(k)
    multiplicaciones, y todos los números se mantienen menores que m.
    """
    resultado, base, i = 1, a % m, 0
    while k > 0:
        if k % 2 == 1:
            resultado = resultado * base % m
            if traza:
                print(f"  usa {a}^{2**i} ≡ {base}  ->  resultado {resultado}")
        elif traza:
            print(f"      {a}^{2**i} ≡ {base}")
        base = base * base % m
        k //= 2
        i += 1
    return resultado

In [18]:
print(potencia_mod(15, 35, 19, traza=True))
print()
print(potencia_mod(7, 29, 19, traza=True))
print(pow(15, 35, 19), pow(7, 29, 19))

  usa 15^1 ≡ 15  ->  resultado 15
  usa 15^2 ≡ 16  ->  resultado 12
      15^4 ≡ 9
      15^8 ≡ 5
      15^16 ≡ 6
  usa 15^32 ≡ 17  ->  resultado 14
14

  usa 7^1 ≡ 7  ->  resultado 7
      7^2 ≡ 11
  usa 7^4 ≡ 7  ->  resultado 11
  usa 7^8 ≡ 11  ->  resultado 7
  usa 7^16 ≡ 7  ->  resultado 11
11
14 11


**Pequeño teorema de Fermat.** Si $p$ es primo y $p\nmid a$, entonces $a^{p-1}\equiv1\bmod p$. Así, $7^{29}\equiv7^{29 \bmod 18}=7^{11}\bmod 19$. Para un número compuesto puede fallar, y eso sirve como test de primalidad (aunque hay números, como $561$, que lo engañan).

In [19]:
p = 19
print(all(pow(a, p - 1, p) == 1 for a in range(1, p)))
print(pow(7, 29 % 18, 19))
print([n for n in range(3, 2000, 2) if pow(2, n - 1, n) == 1 and any(n % d == 0 for d in range(3, int(n**0.5) + 1, 2))])

True
11
[341, 561, 645, 1105, 1387, 1729, 1905]


Los números de la última lista son compuestos y, sin embargo, cumplen $2^{n-1}\equiv1 \bmod n$ (pseudoprimos en base 2).

La diferencia de tiempo con calcular la potencia entera y luego reducir:

In [20]:
import time
a, k, m = 123456789, 3 * 10**5, 10**9 + 7
t0 = time.perf_counter(); r1 = a**k % m; t1 = time.perf_counter()
r2 = potencia_mod(a, k, m); t2 = time.perf_counter()
print(r1 == r2, f"potencia entera {t1 - t0:.3f} s | cuadrados sucesivos {t2 - t1:.6f} s")

True potencia entera 1.260 s | cuadrados sucesivos 0.000093 s


## 6. Sistemas de congruencias

### Teorema chino de los restos

Con módulos primos entre sí dos a dos: $x=\sum b_iM_iy_i$, donde $M_i=M/m_i$ e $y_i$ es el inverso de $M_i$ módulo $m_i$.

In [21]:
def chino(restos, modulos):
    """
    Teorema chino de los restos (5.40): módulos primos entre sí dos a dos.
    Construcción: x = sum b_i M_i y_i, con M_i = M / m_i e y_i el inverso
    de M_i módulo m_i. Devuelve (x, M) con la solución [x]_M.
    """
    M = 1
    for m in modulos:
        M *= m
    x = 0
    for b, m in zip(restos, modulos):
        Mi = M // m
        x += b * Mi * inverso(Mi, m)
    return x % M, M

In [22]:
x, M = chino([2, 4, 1], [3, 5, 7])
print(x, M, [x % 3, x % 5, x % 7])

29 105 [2, 4, 1]


### Teorema chino generalizado y observación 5.42

Se combinan las congruencias de dos en dos; si dos son incompatibles, no hay solución.

In [23]:
def combinar(b1, m1, b2, m2):
    """
    Une x ≡ b1 (mod m1) y x ≡ b2 (mod m2) en una sola congruencia
    módulo mcm(m1, m2), o devuelve None si son incompatibles (5.41).
    """
    d = euclides(m1, m2)
    if (b2 - b1) % d != 0:
        return None
    # x = b1 + m1 t  con  m1 t ≡ b2 - b1 (mod m2)
    t, _ = congruencia_lineal(m1, b2 - b1, m2)
    mcm = m1 // d * m2
    return (b1 + m1 * t) % mcm, mcm


def chino_generalizado(restos, modulos):
    """Teorema chino generalizado (5.41): se combinan las congruencias de dos en dos."""
    b, m = restos[0] % modulos[0], modulos[0]
    for b2, m2 in zip(restos[1:], modulos[1:]):
        r = combinar(b, m, b2, m2)
        if r is None:
            return None
        b, m = r
    return b, m


def sistema(ecuaciones):
    """
    Observación 5.42: resuelve un sistema a_i x ≡ b_i (mod m_i). Primero
    cada ecuación se pasa a x ≡ t_i (mod m_i'), y luego se combinan.
    ecuaciones: lista de ternas (a, b, m).
    """
    restos, modulos = [], []
    for a, b, m in ecuaciones:
        s = congruencia_lineal(a, b, m)
        if s is None:
            return None
        restos.append(s[0])
        modulos.append(s[1])
    return chino_generalizado(restos, modulos)

In [24]:
print(chino_generalizado([1, 3, 4], [6, 4, 9]))
print(sistema([(4, 4, 6), (10, 4, 8), (7, 21, 35)]))
print(chino_generalizado([1, 2], [4, 6]))

(31, 36)
(58, 60)
None


Comprobación por fuerza bruta:

In [25]:
print([x for x in range(36) if x % 6 == 1 and x % 4 == 3 and x % 9 == 4])
print([x for x in range(60) if (4 * x - 4) % 6 == 0 and (10 * x - 4) % 8 == 0 and (7 * x - 21) % 35 == 0])

[31]
[58]


## Referencias

- Grimaldi, R. P. (1998). *Matemática discreta y combinatoria* (3ª ed.). Addison Wesley Longman.
- Rosen, K. H. (2004). *Matemática discreta y sus aplicaciones* (5ª ed.). McGraw-Hill.